# SI4006 · M4 · Amparo — **El sistema ya ve** · Pipeline multimodal

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 4 · Entrega M4

---

Patrón **input**: el usuario adjunta la **foto de un documento jurídico** y un VLM
(**Qwen2-VL-2B**) lo lee y extrae **hechos en JSON**. Esos hechos se concatenan al
relato y entran al sistema de M3 de Amparo, que decide el mecanismo con su motor de
reglas. **M3 no se reescribe**: `ver()` es un traductor en la puerta.

> El VLM **extrae hechos del documento, no decide derecho** — es el mismo principio
> de Amparo ("el LLM extrae, las reglas deciden"), ahora con una imagen en la entrada.

La lógica vive en el paquete: `amparo/multimodal/vlm.py` (el VLM), `amparo/multimodal/enchufe.py`
(el M3 real), `amparo/multimodal/criterios.py` y `scorecard.py` (el harness) y
`amparo/pipelines/m4_multimodal.py` (estas etapas). Este notebook solo las llama.

**Créditos:** T1–T2 Santiago (setup, `cargar()`, `ver()` + válvula) · T3–T4 Lucas (imágenes
sintéticas y `casos_m4.csv`) · T5–T6 Nico (harness, scorecard y tabla de delta) · T7 Martín/Tomás
(el M3 real enchufado: brazo `4 · agente + reglas`).

Corre en Colab con GPU **T4** y necesita el secreto `GROQ_API_KEY`. No ejecutar en local sin GPU.

## 0 · Setup (T1)

Debe correr desde cero en un Colab limpio: **Entorno → Reiniciar y ejecutar todo**.

In [ ]:
# groq y sentence-transformers son del M3 real (T7): el agente llama a Groq y el indice
# de M3 usa un codificador denso y un reranker.
#
# `einops` NO se instala a proposito: el reranker preferido de M3 (jina) lo necesita y,
# si falta, M3 cae a bge-reranker-base, igual que en el notebook de M3. Instalarlo solo
# aqui podria darle a M4 un reranker distinto del de M3. La etapa del enchufe imprime cual cargo.
%pip install -q "transformers>=4.45" accelerate pillow qwen-vl-utils pandas groq sentence-transformers
print('Listo.')

In [ ]:
# Colab arranca vacio: se clona el repo (idempotente). Si el notebook ya corre dentro del
# repo (local) usa esa copia. La raiz es la carpeta que contiene el paquete `amparo/`.
import os, sys, subprocess
from pathlib import Path

REPO_URL, RAMA = 'https://github.com/TomasPosada0626/Amparo.git', 'main'
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'amparo' / 'nucleo' / 'schema.py').exists()), None)
if RAIZ is None:
    if not Path('Amparo').is_dir():
        subprocess.run(['git', 'clone', '-q', '-b', RAMA, REPO_URL, 'Amparo'], check=True)
    RAIZ = Path('Amparo').resolve()
sys.path.insert(0, str(RAIZ))

from amparo.pipelines import m4_multimodal as m4
ctx = m4.preparar()          # raiz, rama y commit, clave de Groq (para si falta)

## 1 · El patrón y las imágenes (T1)

Patrón **input** (`m4/README_PLAN.md`). La pregunta del usuario que necesita ojos: *"adjunto
este documento, ¿qué mecanismo constitucional me corresponde?"*. Las imágenes son
**documentos jurídicos sintéticos** (T3) de `m4/eval_set/imagenes/`, con la etiqueta visible
"documento sintético / generado" (`ANONIMIZACION.md`). Aquí se cargan tres para probar `ver()`.

In [ ]:
m4.cargar_imagenes(ctx);

## 2 · La función `ver()` (T2)

**imagen → (texto_crudo, JSON)**. Las tres reglas del patrón input (S13): **válvula** (si no se
aprecia, `null`), **JSON validado** (si no parsea, el caso es fallo) y, opcional, describir antes
de preguntar (aquí una sola llamada).

In [ ]:
m4.cargar_modelo(ctx);

### 2D · Probar `ver()` sola, sobre 3 imágenes, con latencia (T2)

In [ ]:
m4.probar_ver(ctx);

## 3 · El enchufe: `sistema_multimodal(pregunta, imagen)` (T7 · Martín/Tomás)

El **M3 real**: brazo **`4 · agente + reglas`**, armado igual que en `notebooks/M3_rag.ipynb`.
Es el mismo patrón que M4 (extraer hechos, no decidir derecho), cabe en una T4 junto al VLM
porque razona por API, y es el que mejor aguanta las trampas. **No** es el mejor brazo en el
agregado (ese es el 1): la tabla de delta se compara contra la fila del brazo 4.

In [ ]:
m4.enchufar_m3(ctx);

## 4 · Harness, scorecard y tabla de delta (T6)

`m4/eval_set/casos_m4.csv` (T4): cada caso con su criterio escrito **antes** de correr y uno
adversarial (`valvula`); qué mide cada caso está en `m4/eval_set/CASOS_M4.md`. `correr_t6` valida
el CSV y las imágenes antes del primer caso y guarda `m4/scorecard_m4.csv`, `m4/trazas_m4.json` y,
con los números publicados de M3 (`results/`), `m4/delta_m4.json`.

In [ ]:
m4.validar_eval_set(ctx)
display(m4.tabla_casos(ctx))

In [ ]:
m4.miniaturas(ctx);

In [ ]:
res = m4.correr_harness(ctx)

In [ ]:
di = m4.delta_imagen(ctx)   # misma pregunta con y sin imagen, mismo M3

In [ ]:
display(m4.resumen())

In [ ]:
m4.descargar();   # baja las salidas para subirlas al repo